In [ ]:
!git clone https://github.com/SakanaAI/ShinkaEvolve

In [ ]:
%cd ShinkaEvolve

In [ ]:
%mkdir -p tasks/triple_max

In [ ]:
!pip install -q -e .

In [ ]:
import os

with open("../openai_api_key", "r") as file:
    openai_api_key = file.read().strip()

os.environ["OPENAI_API_KEY"] = openai_api_key

In [ ]:
%mkdir -p tasks/additive_energy_free_n

# Additive-Energy Minimization with Shinka

In this notebook we set up and run an evolutionary search (via `shinka`) to construct integer sequences
$$
\mathbf{a} = (a_1, \dots, a_n) \in \mathbb{Z}_{\ge 0}^n
$$
that satisfy the budget constraint
$$
\sum_{i=1}^n a_i = n^2
$$
and that **minimize the (ordered) additive energy density**
$$
\frac{E(\mathbf{a})}{n^4},
\qquad
E(\mathbf{a}) := \#\{(i,j,k,\ell)\in [n]^4 : a_i + a_j = a_k + a_\ell\}.
$$

\textbf{Intuition.} The additive energy $E(\mathbf{a})$ counts how many ordered quadruples of indices produce the same sum. If many such coincidences occur, the sequence has a lot of additive structure. Under the fixed “mass” constraint $\sum a_i = n^2$, we want to push the sequence toward having \emph{fewer} such coincidences — i.e. we want its pairwise sums to be as spread out as possible.

Equivalently, instead of minimizing
$$
\frac{E(\mathbf{a})}{n^4},
$$
we can \textbf{maximize} the fitness
$$
-\frac{E(\mathbf{a})}{n^4},
$$
so “fewer collisions of sums” $\Rightarrow$ “higher fitness”.

\textbf{Notebook structure.}
1. Restate the optimization problem in code-friendly form.
2. Implement a helper to compute the exact additive energy $E(\mathbf{a})$.
3. Implement the evaluator (same logic as `tasks/additive_energy_free_n/evaluate.py`): shape checks, budget check, fitness.
4. Provide an initial valid constructor (same idea as `initial.py`).
5. Show how to plug this into a `shinka` evolutionary run.
6. (Optional) Read back the results that `shinka` wrote to disk and inspect the best sequence it found.

We will keep all math in `$...$` / `$$...$$` so that it renders correctly in Jupyter.

In [ ]:
# Cell 1: helper to compute additive energy for a list of nonnegative ints

from collections import Counter
from typing import List

def additive_energy(a: List[int]) -> int:
    """
    E(a) = #{(i, j, k, l): a[i] + a[j] = a[k] + a[l]}
         = sum_s r(s)^2
    where r(s) = # of ordered pairs (i, j) with a[i] + a[j] = s.
    """
    n = len(a)
    counts = Counter()
    for i in range(n):
        ai = a[i]
        for j in range(n):
            counts[ai + a[j]] += 1
    return sum(c * c for c in counts.values())

def additive_energy_density(a: List[int]) -> float:
    """
    density = E(a) / n^4
    """
    n = len(a)
    E = additive_energy(a)
    return E / float(n ** 4)


# quick sanity check: the trivial sequence from your initial.py
if __name__ == "__main__":
    n = 8
    a = [n] * n   # sum == n^2
    E = additive_energy(a)
    dens = additive_energy_density(a)
    print(f"n = {n}")
    print(f"a = {a}")
    print(f"E(a) = {E}")
    print(f"density = {dens}")

## Define the Evaluation Function

Now we implement the evaluation logic that checks the validity of a proposed sequence and computes its score.  
This mirrors the file `tasks/additive_energy_free_n/evaluate.py`, but here we’ll keep it self-contained for experimentation.

The evaluator should:
- Verify that the output has the correct structure: a dictionary (or tuple) with integer `n` and list `a`.
- Ensure all entries are nonnegative integers.
- Check the constraint $\sum_i a_i = n^2$.
- Compute the additive energy $E(\mathbf{a})$ and its normalized density $E(\mathbf{a}) / n^4$.
- Return a fitness (to be **maximized**) equal to $-E(\mathbf{a})/n^4$.

In [ ]:
from typing import Any, Dict

def score_one(output: Dict[str, Any]) -> Dict[str, Any]:
    """
    Compute the combined score for a single output dictionary or tuple.
    Returns a structure with "combined_score" (float) and "public" info.
    """
    try:
        # Normalize format
        if not isinstance(output, dict):
            if isinstance(output, tuple) and len(output) == 2:
                n, a = output
                output = {"n": n, "a": a}
            else:
                return {"combined_score": -1.0, "public": {"error": "unexpected output format"}}

        n = int(output.get("n", -1))
        a = output.get("a", None)

        # Shape & type checks
        if not (isinstance(a, list) and n >= 1 and len(a) == n):
            return {"combined_score": -1.0, "public": {"ok": False, "reason": "bad shape"}}

        # Ensure integer nonnegative entries
        a_int = []
        for x in a:
            xi = int(x)
            if xi < 0:
                return {"combined_score": -1.0, "public": {"ok": False, "reason": "negative entry"}}
            a_int.append(xi)

        # Check constraint sum(a) == n^2
        s = sum(a_int)
        if s != n * n:
            return {"combined_score": -1.0, "public": {"ok": False, "reason": "sum(a)!=n^2", "sum": s, "n2": n * n}}

        # Compute additive energy & density
        E = additive_energy(a_int)
        density = E / float(n ** 4)
        score = -density  # we maximize the negative

        return {
            "combined_score": float(score),
            "public": {
                "ok": True,
                "n": n,
                "sum_a": s,
                "energy": E,
                "density": density,
                "fitness": score,
            },
        }

    except Exception as e:
        return {"combined_score": -1.0, "public": {"ok": False, "error": str(e)}}


# Test on the trivial candidate
result = score_one({"n": 8, "a": [8] * 8})
result

## Initial Candidate Generator

The initial solution we take is

$$
n = 8, \quad a = [n, n, \dots, n] \text{ (length } n),
$$

which automatically satisfies
$$
\sum_{i=1}^n a_i = \sum_{i=1}^n n = n \cdot n = n^2.
$$

This is a perfectly valid starting point for evolution, even if it’s not energy-optimal.  
We’ll reproduce that here as a simple function `build_sequence()`, and we’ll immediately score it using the evaluator from the previous cell.

In [ ]:
# Cell 3: initial candidate, identical logic to initial.py

def build_sequence():
    """
    Return a trivial valid sequence:
      n = 8
      a = [8, 8, ..., 8]
    """
    n = 8
    a = [n] * n
    return n, a


# let's build and score it
n0, a0 = build_sequence()
score0 = score_one({"n": n0, "a": a0})

print("Initial candidate:")
print("n =", n0)
print("a =", a0)
print("score info =", score0)

In [ ]:
# Cell 4: Evolution runner setup (do not run yet)

from shinka.core import EvolutionRunner, EvolutionConfig
from shinka.database import DatabaseConfig
from shinka.launch import LocalJobConfig

TASK_SYS_MSG = """You write Python inside EVOLVE-BLOCK only.
Produce a function build_sequence() that returns (n, a) where:
  • n is an integer ≥ 1,
  • a is a list of length n of nonnegative integers,
  • sum(a) == n**2,
  • aim to MINIMIZE additive energy density: E(a)/n**4,
    where E(a) = number of quadruples (i,j,k,l) with a_i + a_j = a_k + a_l.

Be careful:
  • Return JSON-serializable ints (no numpy types).
  • Keep n modest (e.g., 4 ≤ n ≤ 64) to allow O(n^2) evaluation.
  • Avoid trivial or invalid solutions; invalid gets a huge penalty.
"""

# Local job: this points to our evaluator script
job = LocalJobConfig(
    eval_program_path="tasks/additive_energy_free_n/evaluate.py",
    extra_cmd_args={}
)

# Evolution configuration: where the main logic runs
evo = EvolutionConfig(
    init_program_path="tasks/additive_energy_free_n/initial.py",
    num_generations=12,
    max_parallel_jobs=1,
    llm_models=["gpt-4.1-mini"],
    task_sys_msg=TASK_SYS_MSG,
)

# Database configuration: 2 islands, archive of 30 best programs
db = DatabaseConfig(num_islands=2, archive_size=30)

You can now run the evolution loop exactly as in the standalone script:

```python
EvolutionRunner(evo_config=evo, job_config=job, db_config=db).run()

In [ ]:
EvolutionRunner(evo_config=evo, job_config=job, db_config=db).run()

## Inspecting saved Shinka results

We will:
1. Point to the results directory (e.g. `results_20251104_1528...`).
2. Walk all subfolders (`gen_0`, `gen_1`, `best`, ...).
3. Collect JSON files that look like evaluation outputs.
4. Put them into a small table sorted by score.

You can change the path if you run another evolution later.

In [ ]:
# Cell 7a: list and load JSON results from a Shinka run

import os
import json
from pathlib import Path

def collect_results(results_root: str):
    """
    Walk through a shinka results directory and collect all JSON files
    that look like evaluation outputs.
    Returns a list of dicts.
    """
    root = Path(results_root).expanduser()
    all_rows = []

    if not root.exists():
        print(f"Path does not exist: {root}")
        return all_rows

    for path in root.rglob("*.json"):
        try:
            with open(path, "r") as f:
                data = json.load(f)
        except Exception:
            continue

        # We expect something like {"combined_score": ..., "public": {...}}
        row = {
            "file": str(path.relative_to(root)),
            "combined_score": data.get("combined_score", None),
        }

        pub = data.get("public", {})
        if isinstance(pub, dict):
            # add a few useful public fields if present
            for key in ["n", "sum_a", "energy", "density", "fitness", "ok", "reason"]:
                if key in pub:
                    row[key] = pub[key]

        all_rows.append(row)

    return all_rows

# CHANGE THIS to your actual folder name
RESULTS_DIR = "~/Desktop/projects/AI_MATH_2025/ai-in-math-course/Lecture11/ShinkaEvolve/results_20251104_152855"

rows = collect_results(RESULTS_DIR)
print(f"Collected {len(rows)} rows.")
rows[:20]

In [ ]:
# Cell 8a: clean DataFrame and show best results

import pandas as pd

df = pd.DataFrame([r for r in rows if r.get("combined_score") is not None])
df["generation"] = df["file"].str.extract(r"(gen_\d+)")  # extract generation name if present

# sort by generation then by score (descending)
df = df.sort_values(by=["generation", "combined_score"], ascending=[True, False])

print("Best 10 results:")
display(df.head(10)[["file", "generation", "n", "energy", "density", "combined_score"]])

best = df.loc[df["combined_score"].idxmax()]
print("\n🏆 Best candidate:")
print(best.to_string())

In [ ]:
# Cell 8b: visualize energy density over generations

import matplotlib.pyplot as plt

# Aggregate best score per generation
gen_best = (
    df.groupby("generation")["combined_score"]
    .max()
    .reset_index()
    .sort_values("generation")
)

plt.figure(figsize=(6, 4))
plt.plot(gen_best["generation"], -gen_best["combined_score"], marker="o")
plt.title("Additive energy density (lower is better)")
plt.xlabel("Generation")
plt.ylabel("Energy density")
plt.xticks(rotation=45)
plt.grid(True)
plt.show()

In [ ]:
from pathlib import Path

RESULTS_DIR = Path(
    "~/Desktop/projects/AI_MATH_2025/ai-in-math-course/Lecture11/ShinkaEvolve/results_20251104_152855"
).expanduser()

best_main = RESULTS_DIR / "best" / "main.py"

if best_main.exists():
    with open(best_main, "r") as f:
        src = f.read()
    print("Loaded:", best_main)
    print("-" * 60)
    print(src)
else:
    print("main.py not found at:", best_main)

In [ ]:
# Cell: run evolved main.py and inspect the output sequence

import runpy

if best_main.exists():
    ns = runpy.run_path(str(best_main))
    if "build_sequence" in ns:
        n, a = ns["build_sequence"]()
        print("✅ Extracted sequence:")
        print("n =", n)
        print("a =", a)
        print("sum(a) =", sum(a))
    else:
        print("No build_sequence() function found in", best_main)
else:
    print("main.py not found.")

In [ ]:
# Cell: verify energy and score for the extracted sequence

if best_main.exists() and "a" in locals():
    result = score_one({"n": n, "a": a})
    print("Verification:")
    print(result["public"])